# MLflow 3: tracking an ensemble

An **ensemble** combines several models into one: here, three different models each give a risk, and the
ensemble uses their **average** ("soft voting"). The clinic deploys the ensemble, but you still want to know
how each member performs, because when the ensemble gets worse the first question is *which member changed?*

The MLflow pattern for this is **nested runs**:

| Run | Logs |
|---|---|
| **parent run**: the ensemble | its params, its metrics, the ensemble model (the thing you deploy), and which member runs it was built from |
| **child run** per member | that member's params and metrics, measured on the **same test data** so they compare |

In the UI the parent run has a ▸ arrow; open it and the members appear underneath.

Run notebook 1 first (it creates the data). This notebook uses its own experiment, `gam-risk-ensemble`, so it
doesn't mix with the runs you already have.

In [1]:
import json
import os
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
warnings.filterwarnings("ignore", category=UserWarning)

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import skops.io as sio
from mlflow.models import infer_signature
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline

import emr_model as em

pd.set_option("display.width", 160, "display.max_columns", 20)
em.simulate("2024-01", 12)
train = em.load_months("2024-01", "2024-10")       # the same split as notebook 1
test = em.load_months("2024-11", "2024-12")
mlflow.set_experiment("gam-risk-ensemble")
print("tracking URI:", mlflow.get_tracking_uri())

/home/user/mlops-workshop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/10/27 20:12:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/27 20:12:03 INFO mlflow.store.db.utils: Updating database tables


2026/10/27 20:12:06 INFO mlflow.tracking.fluent: Experiment with name 'gam-risk-ensemble' does not exist. Creating a new experiment.


tracking URI: sqlite:////home/user/mlops-workshop/malnutrition/03-mlflow/mlflow.db


## 1. Three different members, one ensemble

Ensembles help most when the members are **different**: a linear model, boosted trees and a random forest
make different mistakes. All three share the same preprocessing (from `emr_model.py`).

In [2]:
preprocess = em.build_model("logistic")[0]          # the shared preprocessing step (impute, scale, one-hot)

members = {
    "logistic": em.build_model("logistic", C=1.0),
    "boosted-trees": em.build_model("gbm", max_depth=3),
    "random-forest": make_pipeline(clone(preprocess),
                                   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
}
member_params = {"logistic": {"model": "logistic", "C": 1.0},
                 "boosted-trees": {"model": "gbm", "max_depth": 3, "learning_rate": 0.1, "n_estimators": 200},
                 "random-forest": {"model": "random_forest", "n_estimators": 300, "min_samples_leaf": 20}}

ensemble = VotingClassifier(list(members.items()), voting="soft")   # soft = average the members' risks
ensemble.fit(train[em.FEATURES], train[em.LABEL])                   # fits every member on the same data
fitted = ensemble.named_estimators_                                  # the fitted members, as used inside it
print("fitted:", list(fitted))

fitted: ['logistic', 'boosted-trees', 'random-forest']


## 2. One parent run, one child run per member

Everything is evaluated on the same test months with the same helper (`em.evaluate`), so the numbers compare.
Two extra things are logged on the parent:

- **`member_runs`**: which member runs this ensemble was built from (its lineage).
- **`roc_auc_without_<member>`**: the ensemble's ROC-AUC if that member were left out. This is how much each
  member *adds*: if leaving one out changes nothing, it isn't earning its place.

In [3]:
def risk(model):
    return model.predict_proba(test[em.FEATURES])[:, 1]


def trusted_types(model):
    # MLflow saves sklearn models with skops and asks which internal types to trust. This is our own model,
    # so we trust what it contains (for a model from elsewhere, read this list first).
    return sio.get_untrusted_types(data=sio.dumps(model))


y = test[em.LABEL].to_numpy()
with mlflow.start_run(run_name="ensemble-soft-vote") as parent:
    mlflow.log_params({"members": ", ".join(members), "voting": "soft",
                       "train_from": "2024-01", "train_until": "2024-10", "test_from": "2024-11", "test_until": "2024-12"})
    mlflow.set_tags({"data_md5": em.fingerprint(train), "git_commit": em.git_commit()})

    member_runs = {}
    for name, model in fitted.items():
        with mlflow.start_run(run_name=name, nested=True) as child:        # nested = a child of the run above
            mlflow.log_params(member_params[name])
            mlflow.set_tags({"role": "ensemble member", "data_md5": em.fingerprint(train)})
            overall, by_group = em.evaluate(model, test)
            mlflow.log_metrics({k: v for k, v in overall.items() if k != "n"})
            mlflow.log_metrics(em.flat_group_metrics(by_group))
            member_runs[name] = child.info.run_id

    overall, by_group = em.evaluate(ensemble, test)
    mlflow.log_metrics({k: v for k, v in overall.items() if k != "n"})
    mlflow.log_metrics(em.flat_group_metrics(by_group))
    mlflow.set_tag("member_runs", json.dumps(member_runs))

    risks = {name: risk(m) for name, m in fitted.items()}
    for left_out in risks:                                               # what does each member add?
        rest = np.mean([r for n, r in risks.items() if n != left_out], axis=0)
        mlflow.log_metric(f"roc_auc_without_{left_out.replace('-', '_')}", roc_auc_score(y, rest))

    info = mlflow.sklearn.log_model(ensemble, name="ensemble",
                                    signature=infer_signature(test[em.FEATURES], ensemble.predict_proba(test[em.FEATURES])),
                                    input_example=test[em.FEATURES].head(3), pyfunc_predict_fn="predict_proba",
                                    skops_trusted_types=trusted_types(ensemble))
print("parent run:", parent.info.run_id)
print("member runs:", member_runs)

parent run: 920175ff681f491189d132cf6577c5ed
member runs: {'logistic': '86d9a4b391a1451c8770a278e272d6a3', 'boosted-trees': 'b5da7416b657404883e933825507c8de', 'random-forest': '700b31fcb42c4e9fbc784e5729b4b3e2'}


## 3. Members and ensemble side by side

`mlflow.search_runs` can pull the children of one parent run: MLflow tags each child with
`mlflow.parentRunId`.

In [4]:
cols = ["tags.mlflow.runName", "metrics.roc_auc", "metrics.sensitivity", "metrics.ppv",
        "metrics.sensitivity_sex_F", "metrics.sensitivity_sex_M"]
kids = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'")[cols]
dad = mlflow.search_runs(filter_string=f"attributes.run_id = '{parent.info.run_id}'")[cols]
table = pd.concat([kids, dad]).set_index("tags.mlflow.runName")
table.columns = [c.split(".", 1)[1] for c in table.columns]
table.round(4)

,roc_auc,sensitivity,ppv,sensitivity_sex_F,sensitivity_sex_M
tags.mlflow.runName,,,,,
random-forest,0.8007,0.5730,0.2600,0.5583,0.585
boosted-trees,0.8038,0.5675,0.2575,0.5460,0.585
logistic,0.8122,0.5702,0.2588,0.5337,0.600
ensemble-soft-vote,0.8083,0.5675,0.2575,0.5337,0.595


In [5]:
without = mlflow.get_run(parent.info.run_id).data.metrics
pd.Series({k.replace("roc_auc_without_", "without "): v for k, v in without.items() if k.startswith("roc_auc_without")},
          name="ensemble ROC-AUC").to_frame().assign(change=lambda d: d["ensemble ROC-AUC"] - without["roc_auc"]).round(4)

,ensemble ROC-AUC,change
without logistic,0.8038,-0.0046
without boosted_trees,0.8091,0.0008
without random_forest,0.8099,0.0016


**Reading it.** On this data the ensemble (ROC-AUC 0.808) is slightly **worse** than logistic regression alone
(0.812). The "without" table says why: leaving out logistic hurts the most (−0.005), while leaving out the boosted
trees or the random forest would *improve* the ensemble a little. Logistic is doing the work; the other two add
complexity (more to retrain, monitor and explain) and nothing else. An ensemble is not automatically better, and
without the member runs you would never have seen this.

**In the UI** (*gam-risk-ensemble* experiment): the `ensemble-soft-vote` row has a ▸ arrow; open it to see the
three member runs. Select all four and **Compare** to see their metrics side by side.

## 4. Register the ensemble, not the members

What the clinic uses is the ensemble, so that is what goes in the registry. The `member_runs` tag keeps the
link to the members, so an auditor can still find out exactly what is inside version 1.

In [6]:
version = mlflow.register_model(info.model_uri, "gam-risk-ensemble")
print("registered gam-risk-ensemble version", version.version)
loaded = mlflow.sklearn.load_model(f"models:/gam-risk-ensemble/{version.version}")
print("members inside the registered model:", list(loaded.named_estimators_))

registered gam-risk-ensemble version 1


Successfully registered model 'gam-risk-ensemble'.
Created version '1' of model 'gam-risk-ensemble'.


members inside the registered model: ['logistic', 'boosted-trees', 'random-forest']


## When it goes wrong in production

Score new months with the members as well as the ensemble, and log each member's metrics to the same
monitoring run. When the ensemble's performance drops, the member table tells you whether one member broke
(for example, the one that leans on MUAC after a MUAC unit change) or all of them did (the world changed).

**Exercise.** Add a fourth member (`em.build_model("gbm", max_depth=2, learning_rate=0.05)`), re-run, and use the
"without" table to decide whether it earns its place.